# Face-count eval — Phase 1: extract candidate frames from the OEP dataset

For issue #1222 (face-count detection accuracy). This notebook does NOT produce
ground truth by itself — it decodes webcam video from the MSU Online Exam
Proctoring (OEP) dataset (real students at a desk, webcam above the monitor —
the actual use case, unlike the general-photography WIDER FACE set used in the
first pass of this eval), samples frames across the timeline, and runs a quick
detector pass *only* to flag candidate rare frames (0-face / 2+-face moments)
worth a human looking at. It outputs **contact sheets** (grids of labeled
thumbnails) so a human can review hundreds of candidates in a few passes and
write the actual ground-truth `labels.csv` by hand afterwards — using the
detector to find candidates for a human to label is fine; using it to label
its own test set would not be (that's circular).

**Before running:** click *Add Data* (top right) → search
`MSU Online Exam Proctoring Dataset` (raajanwankhade/oep-dataset) → add it.
Internet does not need to be on except to `pip install mediapipe` if it isn't
already present in the environment (it usually is).

Outputs (in the Output tab after running):
- `frames/` — full-resolution candidate JPEGs
- `contact_sheets/` — labeled thumbnail grids, ordered so rare (0-face,
  2+-face) candidates come first
- `candidate_manifest.csv`, `sheet_manifest.csv` — bookkeeping, not ground truth

In [ ]:
import os, glob, csv
import cv2
import numpy as np

try:
    import mediapipe as mp
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mediapipe"])
    import mediapipe as mp

print("mediapipe", mp.__version__)
print("opencv", cv2.__version__)

In [ ]:
# Auto-discover the dataset root regardless of exact mount path
INPUT_ROOT = "/kaggle/input"
dataset_root = None
for base, dirs, files in os.walk(INPUT_ROOT):
    if "gt.txt" in files:
        dataset_root = os.path.dirname(base)
        break

if dataset_root is None:
    raise RuntimeError(
        "Could not find the OEP dataset under /kaggle/input. "
        "Add Data -> search 'MSU Online Exam Proctoring Dataset' "
        "(raajanwankhade/oep-dataset) first, then re-run."
    )

print("dataset root:", dataset_root)
print(sorted(os.listdir(dataset_root)))

In [ ]:
# Real students (proctor invokes cheating by talking / walking up to the
# student / handing them a book -- our best real source of genuine
# multi-person frames) prioritized, plus a handful of acting subjects for
# baseline diversity. See dataset_root/READ_ME.txt for the subject-group split.
REAL_SUBJECTS = ["subject10", "subject11", "subject12", "subject13", "subject14",
                  "subject15", "subject16", "subject18", "subject19"]
ACTING_SUBJECTS = ["subject1", "subject5", "subject9", "subject20"]
SUBJECTS = REAL_SUBJECTS + ACTING_SUBJECTS

SAMPLE_EVERY_S = 2.5     # candidate sampling interval along the timeline
MAX_DURATION_S = 900     # cap per video (15 min) to bound notebook runtime
NORMAL_KEEP_EVERY = 4    # subsample the (very common) predicted-1-face candidates

def find_webcam_video(subject_dir):
    \"\"\"Webcam file = <username>1.avi (wearcam is <username>2.avi); username
    is read from the .wav filename since it varies per subject.\"\"\"
    wavs = glob.glob(os.path.join(subject_dir, "*.wav"))
    if not wavs:
        return None
    username = os.path.splitext(os.path.basename(wavs[0]))[0]
    candidate = os.path.join(subject_dir, username + "1.avi")
    if os.path.exists(candidate):
        return candidate
    # fallback: any *1.avi that isn't the wearcam (*2.avi)
    for f in glob.glob(os.path.join(subject_dir, "*1.avi")):
        if not f.endswith("2.avi"):
            return f
    return None

In [ ]:
mp_face = mp.solutions.face_detection
# model_selection=1 = "full range" (5m), matching the frontend's
# modelType: "full" config (frontend/src/components/ai/FaceDetectorWorker.ts).
detector = mp_face.FaceDetection(model_selection=1, min_detection_confidence=0.5)

FRAMES_DIR = "/kaggle/working/frames"
os.makedirs(FRAMES_DIR, exist_ok=True)

manifest = []
kept_normal_counter = 0

for subject in SUBJECTS:
    subject_dir = os.path.join(dataset_root, subject)
    if not os.path.isdir(subject_dir):
        print(f"[{subject}] folder not found, skipping")
        continue
    video_path = find_webcam_video(subject_dir)
    if not video_path:
        print(f"[{subject}] no webcam video found, skipping")
        continue

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    sample_every_frames = max(1, int(round(fps * SAMPLE_EVERY_S)))
    max_frames = int(MAX_DURATION_S * fps)

    frame_idx = 0
    read_failures = 0
    print(f"[{subject}] video={os.path.basename(video_path)} fps={fps:.1f}")

    while frame_idx < max_frames:
        ok, frame = cap.read()
        if not ok or frame is None:
            read_failures += 1
            if read_failures > 5:
                break
            frame_idx += 1
            continue

        if frame_idx % sample_every_frames == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = detector.process(rgb)
            pred_count = len(result.detections) if result.detections else 0

            keep = True
            if pred_count == 1:
                kept_normal_counter += 1
                keep = (kept_normal_counter % NORMAL_KEEP_EVERY == 0)

            if keep:
                t_s = frame_idx / fps
                frame_id = f"oep_{subject}_{int(t_s):05d}s.jpg"
                cv2.imwrite(os.path.join(FRAMES_DIR, frame_id), frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
                manifest.append({
                    "frame_id": frame_id,
                    "subject": subject,
                    "group": "real" if subject in REAL_SUBJECTS else "acting",
                    "timestamp_s": round(t_s, 1),
                    "detector_predicted_count": pred_count,
                })

        frame_idx += 1

    cap.release()
    subj_frames = [m for m in manifest if m["subject"] == subject]
    zero = sum(1 for m in subj_frames if m["detector_predicted_count"] == 0)
    multi = sum(1 for m in subj_frames if m["detector_predicted_count"] >= 2)
    print(f"[{subject}] kept {len(subj_frames)} candidates (0-face: {zero}, 2+-face: {multi})")

print(f"\nTotal candidate frames: {len(manifest)}")

In [ ]:
manifest_path = "/kaggle/working/candidate_manifest.csv"
with open(manifest_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["frame_id", "subject", "group", "timestamp_s", "detector_predicted_count"])
    writer.writeheader()
    writer.writerows(manifest)
print("wrote", manifest_path, "-", len(manifest), "rows")

In [ ]:
from PIL import Image, ImageDraw, ImageFont

CONTACT_DIR = "/kaggle/working/contact_sheets"
os.makedirs(CONTACT_DIR, exist_ok=True)

THUMB = 160
COLS, ROWS = 5, 5
PER_SHEET = COLS * ROWS
LABEL_H = 22

# Rare candidates (0-face, 2+-face) first -- review every one of those.
# The subsampled 1-face pool comes after.
ordered = (
    [m for m in manifest if m["detector_predicted_count"] == 0]
    + [m for m in manifest if m["detector_predicted_count"] >= 2]
    + [m for m in manifest if m["detector_predicted_count"] == 1]
)

def load_font():
    for path in ("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
                 "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"):
        try:
            return ImageFont.truetype(path, 13)
        except Exception:
            continue
    return ImageFont.load_default()

font = load_font()
sheet_manifest = []

for sheet_idx in range(0, len(ordered), PER_SHEET):
    chunk = ordered[sheet_idx:sheet_idx + PER_SHEET]
    sheet = Image.new("RGB", (COLS * THUMB, ROWS * (THUMB + LABEL_H)), "white")
    draw = ImageDraw.Draw(sheet)
    for i, m in enumerate(chunk):
        img = Image.open(os.path.join(FRAMES_DIR, m["frame_id"])).convert("RGB")
        img.thumbnail((THUMB, THUMB))
        col, row = i % COLS, i // COLS
        x, y = col * THUMB, row * (THUMB + LABEL_H)
        sheet.paste(img, (x, y + LABEL_H))
        label = f"{m['subject'][7:]}@{int(m['timestamp_s'])}s p={m['detector_predicted_count']}"
        draw.text((x + 2, y + 3), label, fill="black", font=font)
        sheet_manifest.append({"sheet": f"sheet_{sheet_idx // PER_SHEET:03d}", "position": i, **m})
    sheet_path = os.path.join(CONTACT_DIR, f"sheet_{sheet_idx // PER_SHEET:03d}.jpg")
    sheet.save(sheet_path, quality=88)

with open("/kaggle/working/sheet_manifest.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["sheet", "position", "frame_id", "subject", "group", "timestamp_s", "detector_predicted_count"])
    writer.writeheader()
    writer.writerows(sheet_manifest)

n_sheets = len(os.listdir(CONTACT_DIR))
print(f"wrote {n_sheets} contact sheets covering {len(ordered)} frames")
print(f"  0-face candidates: {sum(1 for m in manifest if m['detector_predicted_count']==0)}")
print(f"  2+-face candidates: {sum(1 for m in manifest if m['detector_predicted_count']>=2)}")
print(f"  1-face candidates (subsampled): {sum(1 for m in manifest if m['detector_predicted_count']==1)}")